# 08. 여러 에이전트 엮기 · 평가 · 성과지표(KPI) (종합편)

이 노트북은 세 가지를 실제로 `gpt-5-nano` 모델에 돌려보며 확인합니다.

- **오케스트레이션**: 여러 개의 에이전트(스스로 판단해 도구를 골라 쓰는 AI)를 어떻게 엮어서 하나의 일을 처리할지
- **평가**: 프롬프트가 얼마나 좋은지 어떻게 점수를 매길지
- **성과지표(KPI, 핵심 성과 지표)**: 무엇을 진짜 성과로 봐야 하는지

아래 팁들을 하나씩 실험으로 검증합니다.

- **팁 3 — 도구가 200개일 때**: 에이전트를 여러 개로 잘게 나누기보다, 하나의 에이전트에게 도구를 다 주고 *호출하기 전에 계획부터* 세우게 한다.
- **팁 4 — 한국어 질문의 작업 분해 문제**: 작업 분해(질문을 여러 하위 작업으로 쪼개는 것)를 억지로 하기보다, 잘 다듬은 시스템 프롬프트로 한 번에 처리하는 편이 낫다.
- **팁 5 — '판단 불가' 경로**: 입력을 분류하는 분류기에는 '어디에도 해당 안 됨' 예외 경로가 꼭 필요하다.
- **팁 22 — 정답지부터 만들기**: 답에 반드시 들어가야 할 핵심 키워드를 미리 정해두고, `keyword_hits`로 프롬프트 출력에 점수를 매긴다.
- **팁 24 — AI가 짠 프롬프트의 한계**: AI가 만든 프롬프트는 서로 비슷비슷하다. 사람이 도메인 지식으로 다듬어야 한다.
- **팁 27 — 여러 턴에 걸친 오류 전파**: 정보가 부족한 요청에 모델이 넘겨짚어 답하면, 그 첫 오답이 이후 대화 전체를 잘못된 전제로 끌고 간다. 그래서 '부족하면 되물어라' 규칙이 필요하다.
- **팁 28 — 진짜 성과지표는 토큰과 응답 시간**: 응답 시간(레이턴시, 요청을 보내고 답이 돌아오기까지 걸리는 시간)을 말한다. 장황한 프롬프트와 압축한 프롬프트가 같은 정확도를 내면서 비용이 얼마나 차이 나는지 직접 잰다.
- **팁 29 — AI 산출물 특유의 티**: AI가 만든 결과물은 그대로 쓰지 말고 사람이 마무리해야 한다.
- **팁 31 — 여러 턴을 한 번의 요청으로 압축**: 지난 대화를 다 버리고, '지금까지의 상태를 요약한 마크다운 + 최신 명령'만으로 매 턴을 새로 조립한다.
- **팁 37 — 스스로 검토하는 단계**: 답을 내보내기 전에 `<final_review>`(최종 점검) 단계를 강제해, 규칙을 어기지 않았는지 모델이 직접 확인하게 한다.

> 참고: 모델은 `gpt-5-nano`입니다. temperature(출력의 무작위성 정도)가 1로 고정돼 바꿀 수 없어서, 창의적으로 갈지 보수적으로 갈지는 **프롬프트 텍스트로만** 조절합니다. 또 `max_tokens` 대신 `max_completion_tokens`를 씁니다. 이 모델은 내부적으로 추론을 하는 방식이라 토큰 예산이 너무 작으면 정작 본문이 비어버릴 수 있어, 넉넉하게 줍니다.

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## 팁 3 — 도구가 200개면 '나누기'보다 '계획하기'

도구가 수십, 수백 개라고 해서 에이전트를 잘게 나누면(입력을 나눠 보내는 라우터 → 하위 에이전트 → 그 아래 또 하위 에이전트…) 이들을 엮고 조율하는 비용, 응답 지연, 그리고 한 곳의 오류가 다음 단계로 번지는 문제가 함께 커진다. 실무에서 더 잘 통하는 방향은 **하나의 에이전트**에게 도구 목록을 전부 주되, *실제로 도구를 부르기 전에* `<scratchpad>`(생각을 적어두는 메모 공간, Think 단계)에서 **어떤 도구를 왜, 어떤 순서로** 부를지 계획을 먼저 세우게 하는 것이다.

**검증 방법**: 도구 목록(요약)을 시스템 프롬프트로 주고, 모델에게 '아직 도구를 부르지 말고 `<scratchpad>`에 호출 계획만 적어라'라고 시킨다. 실제 도구 호출 없이도 **계획을 먼저 뽑아내는 방식**이 되는지 관찰한다.

In [ ]:
# Tip 3: 단일 에이전트에게 '호출 전 계획(Think/Scratchpad)'을 먼저 세우게 한다
# (실제 도구 실행 없이, 계획 텍스트만 뽑아내는 패턴)
TOOLS = [
    "search_orders(customer_id)  # 고객 주문 조회",
    "get_refund_policy(sku)      # SKU별 환불정책",
    "issue_refund(order_id, amt) # 환불 실행",
    "notify_customer(msg)        # 고객 알림 발송",
    "... (총 200여 개 중 4개만 발췌) ...",
]
sys_p = (
    "너는 단일 CS 에이전트다. 아래 도구 카탈로그를 쓸 수 있다:\n"
    + "\n".join(f"- {t}" for t in TOOLS)
    + "\n\n규칙: 도구를 곧바로 호출하지 말 것. 먼저 <scratchpad> 안에"
    " (1) 사용자 의도 (2) 필요한 도구를 '호출 순서'대로 (3) 각 호출에 필요한 입력값의 출처"
    " 를 번호로 계획하라. </scratchpad> 뒤에 한 줄 요약만 붙여라."
)
req = "주문번호 A-1023 환불해줘. 이미 배송 시작됐는지도 확인하고, 되면 처리하고 나한테 알림도 줘."
print(ask(req, system=sys_p, reasoning_effort="minimal", max_completion_tokens=700))

## 팁 4 — 억지로 나누기 vs 잘 다듬은 시스템 프롬프트로 한 번에 처리하기

복잡한 한국어 질문을 만나면 '작업 분해 → 하위 작업별로 따로 호출 → 결과 합치기'로 가고 싶어진다. 하지만 한국어처럼 앞뒤 맥락에 많이 기대는 질문은 억지로 쪼개면 **나눈 경계에서 미묘한 뉘앙스가 사라지고** 호출 횟수만 늘어난다. 역할, 형식, 제약을 촘촘히 적어둔 **시스템 프롬프트 하나로 한 번에(single-shot, 한 번의 요청으로) 처리**하는 편이 더 일관된 답을 줄 때가 많다.

**검증 방법**: 같은 복합 질문을 (A) 하위 질문으로 강제로 나눠 각각의 답을 이어 붙이는 방식과 (B) 잘 다듬은 시스템 프롬프트로 한 번에 처리하는 방식으로 각각 돌려, 두 결과가 얼마나 하나의 맥락으로 잘 이어지는지 비교한다.

In [ ]:
# Tip 4: 강제 분해(A) vs 탄탄한 시스템 프롬프트 single-shot(B)
q = (
    "우리 카페 신메뉴 '흑임자 라떼'를 20대 타깃으로 출시하려는데, "
    "가격 전략, 인스타 홍보 문구 1개, 예상 리스크 1개를 알려줘."
)

# (A) 억지 분해: 하위질문을 따로따로 물어보고 이어붙이기 (분해 경계에서 맥락 단절)
subs = ["가격 전략만 알려줘.", "인스타 홍보 문구 1개만 써줘.", "예상 리스크 1개만 알려줘."]
a_parts = [ask(s, reasoning_effort="minimal", max_completion_tokens=250) for s in subs]
a = "\n".join(f"[{s}]\n{p}" for s, p in zip(subs, a_parts))

# (B) 탄탄한 시스템 프롬프트로 한 방에 (맥락을 공유한 채 일관되게)
sys_b = (
    "너는 F&B 브랜드 마케터다. 항상 '하나의 캠페인'이라는 일관된 맥락 아래 답하라."
    " 출력은 정확히 세 줄: '가격:', '홍보:', '리스크:' 로 시작하고 각 한 문장."
)
b = ask(q, system=sys_b, reasoning_effort="minimal", max_completion_tokens=400)

compare("(A) 강제 분해 후 이어붙임", a, "(B) 탄탄한 시스템 프롬프트 single-shot", b)

## 팁 5 — 분류기에는 '판단 불가(Unclassified)' 경로가 꼭 있어야 한다

입력을 어디로 보낼지 정하는 분류기에 유효한 라벨만 나열해 두면, 애매하거나 범위를 벗어난 입력에도 모델이 **그중 하나를 억지로 고른다**. 이렇게 잘못 분류하면 그 뒤 처리(다운스트림, 분류 결과를 받아 이어지는 이후 단계) 전체가 잘못된 경로로 흘러간다. 그래서 라벨 목록에는 항상 `Unclassified`(또는 '기타/판단 불가')라는 **예외 경로**를 넣어, 확신이 낮을 때는 사람이나 상위 처리로 넘기도록(escalate) 만들어야 한다.

**검증 방법**: 명백히 범위 밖인 애매한 입력을 (A) 라벨 2개만 주는 분류기와 (B) `Unclassified`를 추가한 분류기에 각각 `ask_json`으로 넣어, 억지 분류와 안전한 예외 처리를 비교한다.

In [ ]:
# Tip 5: 옵션 2개만(A) vs +Unclassified(B) — 애매 입력에 대한 반응 비교
ambiguous = "내일 날씨 어때? 그리고 우리 강아지 밥은 언제 줘야 해?"  # 결제/배송 어디에도 안 맞음

# (A) 유효 라벨 2개만 -> 억지로 하나를 고를 위험
sys_a = (
    "입력을 다음 중 하나로 분류해 JSON {\"label\": ...} 로만 답하라."
    " 가능한 label: 'billing'(결제), 'shipping'(배송)."
)
res_a = ask_json(ambiguous, system=sys_a, reasoning_effort="minimal", max_completion_tokens=200)

# (B) 안전 탈출 라우트 추가
sys_b = (
    "입력을 다음 중 하나로 분류해 JSON {\"label\": ..., \"confidence\": 0~1} 로만 답하라."
    " 가능한 label: 'billing'(결제), 'shipping'(배송),"
    " 'Unclassified'(둘 다 아니거나 확신이 낮으면 반드시 이걸 선택)."
)
res_b = ask_json(ambiguous, system=sys_b, reasoning_effort="minimal", max_completion_tokens=200)

print("애매 입력:", ambiguous)
print("(A) 옵션 2개만        ->", res_a)
print("(B) +Unclassified 라우트 ->", res_b)

## 팁 22 — 채점 먼저: 정답지(핵심 키워드)를 정하고 프롬프트에 점수 매기기

'왠지 이게 더 좋아 보인다'로 프롬프트를 고르면 다음에 같은 판단을 다시 재현하기 어렵다. 먼저 **정답에 반드시 등장해야 할 핵심 키워드와 수치 목록**을 정답지로 고정하고, 각 프롬프트의 출력이 그 키워드를 몇 개나 맞혔는지 `keyword_hits`로 점수를 매겨 **객관적으로** 비교한다.

**검증 방법**: 같은 질문(HTTP 상태 코드 설명)에 대해 (A) 막연한 프롬프트와 (B) 정답지를 겨냥해 요구사항을 구체적으로 적은 프롬프트를 각각 돌리고, 미리 만든 키워드 정답지로 두 출력을 채점한다.

In [ ]:
# Tip 22: 정답지(키워드)로 두 프롬프트를 객관 채점
GOLD = ["200", "301", "404", "500", "redirect"]  # 이 답변에 반드시 나와야 할 핵심들(정답지)

# (A) 막연한 프롬프트
a = ask("HTTP 상태코드에 대해 설명해줘.", reasoning_effort="minimal", max_completion_tokens=350)

# (B) 정답지를 겨냥해 커버리지를 요구하는 프롬프트
b = ask(
    "HTTP 상태코드를 설명하되, 반드시 200, 301, 404, 500을 각각 언급하고 "
    "301이 redirect(리다이렉트)라는 점을 포함해라.",
    reasoning_effort="minimal", max_completion_tokens=350,
)

sa, sb = keyword_hits(a, GOLD), keyword_hits(b, GOLD)
print(f"(A) 막연  점수: {sa['score']}/{sa['total']}  {sa['hits']}")
print(f"(B) 겨냥  점수: {sb['score']}/{sb['total']}  {sb['hits']}")
compare("(A) 막연한 프롬프트", a, "(B) 정답지 겨냥 프롬프트", b)

## 팁 24 — AI가 짠 프롬프트의 한계: 서로 비슷비슷하다

'프롬프트를 짜줘'라고 하면 모델은 **정형화된 틀**(역할 부여 → 단계 나열 → '전문적으로', '명확하게' 같은 흔한 표현 → 이모지 불릿)을 내놓는다. 출발점으로는 쓸 만하지만, 그 도메인에만 있는 특수한 제약, 예외 상황, 원하는 말투는 빠져 있다. **AI가 만든 프롬프트는 초안일 뿐이고, 사람이 도메인 지식으로 마무리**해야 한다.

**검증 방법**: 모델에게 프롬프트를 하나 짜게 시킨 뒤, 그 결과물에서 '어디서나 나올 법한 신호'(흔한 표현, 뻔한 일반론, 빠진 제약)를 직접 눈으로 확인한다.

In [ ]:
# Tip 24: 모델에게 프롬프트를 짜게 시키고 '천편일률' 신호를 관찰
meta = ask(
    "'고객 이메일에 답장하는 챗봇'을 위한 시스템 프롬프트를 하나 작성해줘.",
    reasoning_effort="minimal", max_completion_tokens=500,
)
print(meta)

# 상투어/일반론 신호를 러프하게 카운트 (많을수록 '템플릿 냄새')
cliche = ["전문", "명확", "친절", "정확", "공손", "professional", "helpful", "clear", "step"]
hit = keyword_hits(meta, cliche)
print("\n[관찰] 상투어 신호:", hit["score"], "/", hit["total"], "->",
      [k for k, v in hit["hits"].items() if v])
print("[해석] 도메인 특수 제약(SLA 시간, 환불 한도, 금칙어 등)이 비어 있다면 사람이 채워야 함.")

## 팁 27 — 여러 턴에 걸친 오류 전파: 첫 오답이 대화를 계속 잘못 끌고 간다

정보가 모자란 요청에 모델이 **넘겨짚어(부족한 정보를 스스로 지어내) 답하면**, 그 첫 오답이 대화 기록(컨텍스트)에 남아 이후 턴들을 계속 잘못된 전제 위에서 진행하게 만든다. 여러 턴에 걸친 대화(멀티턴)에서 특히 문제가 된다. 막는 방법은 간단하다. **'정보가 부족하면 넘겨짚지 말고 되물어라'**를 시스템 규칙으로 강제하는 것이다.

**검증 방법**: 핵심 정보가 빠진 모호한 요청을 (A) 그냥 답하게 하는 경우와 (B) '부족하면 되묻기' 규칙을 준 경우로 각각 돌려, **첫 응답**이 넘겨짚은 답인지 되묻는 질문인지 비교한다.

In [ ]:
# Tip 27: 유추(A) vs '부족하면 되묻기'(B) — 첫 응답의 차이
vague = "환불 처리 좀 해줘."  # 주문번호/사유/결제수단 등 필수정보가 전부 없음

# (A) 규칙 없음 -> 넘겨짚어(유추) 답할 위험
a = ask(vague, reasoning_effort="minimal", max_completion_tokens=250)

# (B) 정보 부족 시 되묻기 강제
sys_b = (
    "너는 CS 에이전트다. 요청 처리에 필요한 정보가 하나라도 없으면, 절대 유추하지 말고"
    " 부족한 항목을 콕 집어 되물어라. 확인되기 전에는 처리한 척하지 마라."
)
b = ask(vague, system=sys_b, reasoning_effort="minimal", max_completion_tokens=250)

compare("(A) 규칙 없음(유추 위험)", a, "(B) 부족하면 되묻기", b)

## 팁 28 — 진짜 성과지표는 토큰과 응답 시간 (핵심 실험)

출력 품질이 같다면, 실제 서비스에서 차이를 만드는 건 **비용(프롬프트에 들어가는 토큰 수)과 응답 시간(레이턴시)**이다. 배경 설명을 길게 늘어놓은 프롬프트와, 같은 요구를 짧게 압축한 프롬프트가 **똑같은 정답**을 내면서 토큰 수와 응답 시간이 얼마나 차이 나는지 `ask_meta`로 직접 잰다.

**검증 방법**: 같은 문제(사과 3개 + 과일 바구니 5개는 몇 개?)를 (A) 장황한 프롬프트와 (B) 압축한 프롬프트로 돌리고, `prompt_tokens`(프롬프트 토큰 수)와 `latency`(응답 시간)를 나란히 비교한다. 정답 '8'이 두 경우 모두 나오는지도 함께 확인한다.

In [ ]:
# Tip 28: 장황 vs 압축 — 정확도 유지하며 prompt_tokens/latency 비교 (ask_meta 실측)
verbose = (
    "안녕하세요, 바쁘신데 정말 죄송합니다. 제가 간단한 산수 문제가 하나 있는데요, "
    "혹시 시간 되시면 도와주실 수 있을까요? 상황을 설명드리자면, 제 책상 위에 사과가 "
    "3개 놓여 있고요, 그리고 옆에 과일 바구니가 하나 있는데 그 안에 사과가 5개 들어 "
    "있습니다. 자, 그럼 사과가 총 몇 개인지 계산해 주시면 정말 감사하겠습니다. 답만 숫자로요."
)
compact = "사과 3 + 5 = ? 숫자만."

va = ask_meta(verbose, reasoning_effort="minimal", max_completion_tokens=50)
co = ask_meta(compact, reasoning_effort="minimal", max_completion_tokens=50)

print("[장황] 답:", repr(va["text"]), "| prompt_tokens:", va["prompt_tokens"], "| latency:", va["latency"], "s")
print("[압축] 답:", repr(co["text"]), "| prompt_tokens:", co["prompt_tokens"], "| latency:", co["latency"], "s")
if co["prompt_tokens"]:
    print(f"\n프롬프트 토큰 절감률: {(1 - co['prompt_tokens']/va['prompt_tokens'])*100:.0f}%  "
          "(정답이 둘 다 8이면 '같은 정확도에 반값' 팁이 검증됨)")

## 팁 29 — AI가 만든 발표 개요·구조의 티 (그대로 납품하지 말 것)

AI가 뽑아낸 슬라이드 개요나 문서 구조에는 **눈에 띄는 공통 패턴**이 있다. 항상 3~5개로 개수를 맞춘 불릿, '개요-본론-결론'식 정형 구성, 내용 없이 그럴듯하기만 한 헤드라인, 지나치게 반복되는 병렬 구조 같은 것들이다. 이런 특징은 받아 보는 쪽도 금방 알아챈다. 그러니 **초안을 빠르게 만드는 용도로만 쓰고, 사람이 이야기 흐름과 강약, 구체적인 수치로 마무리**해야 한다.

**검증 방법(가벼운 데모)**: 모델에게 발표 개요를 짜게 시키고, '개수를 맞춘 불릿, 정형 구조, 뻔한 헤드라인' 같은 AI 특유의 신호를 눈으로 확인한다. 이 셀은 '그대로 쓰지 말라'는 걸 직접 느껴보기 위한 관찰용이다.

In [ ]:
# Tip 29: AI 발표 개요의 '티'를 관찰 (초안일 뿐, 사람이 마감해야 함)
outline = ask(
    "'우리 팀 2분기 성과' 사내 발표를 위한 슬라이드 개요를 만들어줘.",
    reasoning_effort="minimal", max_completion_tokens=450,
)
print(outline)

# 러프한 '기계 냄새' 진단: 불릿 개수 균등성 / 상투 헤드라인
lines = [l for l in outline.splitlines() if l.strip()]
bullets = [l for l in lines if l.lstrip().startswith(("-", "*", "•")) or l.lstrip()[:2].strip().isdigit()]
print("\n[관찰] 전체 줄:", len(lines), "| 불릿류 줄:", len(bullets))
print("[체크리스트] 구체 수치가 비었나? 스토리(왜 중요한가)가 빠졌나? 헤드라인이 상투적인가?")
print("           -> 하나라도 YES면 그대로 납품 금지, 사람이 마감해야 함.")

## 팁 31 — 여러 턴을 한 번의 요청으로 압축 (Claude Code의 원리)

긴 대화를 매 턴 통째로 다시 보내면 토큰이 크게 늘고, 오래된 잡담이 지금의 판단을 흐린다. Claude Code 같은 에이전트가 쓰는 핵심 방법은 **지난 대화를 그대로 쌓지 않고, '지금까지의 상태'를 마크다운 요약으로 압축해 덮어쓴 다음, 최신 명령 하나만 붙여 매 턴을 한 번의 요청(원샷)으로 다시 조립**하는 것이다.

**검증 방법**: 대화 기록을 계속 쌓는 대신, `state_md`(상태를 요약한 마크다운)와 `new_command`(최신 명령)를 매 턴 다시 조립하는 함수를 만들어, 짧은 한 번의 요청으로 처리되는 흐름을 보여준다. 대화가 아무리 길어져도 프롬프트는 '상태 + 명령' 형태로 일정하게 유지된다.

In [ ]:
# Tip 31: '상태 마크다운 + 새 명령'을 매 턴 원샷으로 재조립
def one_shot_turn(state_md: str, new_command: str) -> dict:
    """과거 히스토리 대신, 압축된 상태 마크다운 + 최신 명령만으로 원샷 처리."""
    prompt = (
        "# 현재 상태 (누적 요약)\n" + state_md +
        "\n\n# 새 명령\n" + new_command +
        "\n\n위 상태를 반영해 새 명령만 처리하고, 답변 끝에 '갱신된 상태:' 로 시작하는"
        " 3줄 이내 마크다운 상태 요약을 덧붙여라."
    )
    return ask_meta(prompt, reasoning_effort="minimal", max_completion_tokens=400)

# 턴 1: 상태는 짧은 마크다운으로 유지 (지난 잡담은 버림)
state = "- 프로젝트: 카페 앱\n- 확정: 이름 '데일리빈'\n- 미정: 색상 테마"
r1 = one_shot_turn(state, "색상 테마를 따뜻한 톤으로 정해줘.")
print("[턴1] prompt_tokens:", r1["prompt_tokens"])
print(r1["text"])

# 턴 2: '누적 히스토리'가 아니라 '갱신된 상태 마크다운'만 이어받아 다시 원샷
state2 = "- 프로젝트: 카페 앱\n- 확정: 이름 '데일리빈', 색상 따뜻한 톤\n- 미정: 로고 문구"
r2 = one_shot_turn(state2, "로고에 넣을 한 줄 슬로건 하나 지어줘.")
print("\n[턴2] prompt_tokens:", r2["prompt_tokens"], "  <- 히스토리를 안 쌓아 프롬프트가 폭증하지 않음")
print(r2["text"])

## 팁 37 — 스스로 검토하는 단계: 출력 직전에 `<final_review>` 강제

금칙어, 형식, 길이 제한 같은 규칙이 있는 작업에서 모델은 종종 규칙을 어긴 채 바로 답을 내보낸다. 답을 내보내기 **직전에** `<final_review>`(최종 점검) 단계를 강제해 '내 답이 모든 규칙을 지켰는지 스스로 확인하고, 다 지켰을 때만 최종본을 내라'고 시키면 규칙 위반이 눈에 띄게 줄어든다.

**검증 방법**: 명시적인 규칙이 있는 작업을 (A) 검토 단계 없이와 (B) `<final_review>`를 강제한 경우로 각각 돌려 규칙 위반 여부를 비교한다. 여기서는 '숫자를 절대 쓰지 말고, 정확히 세 문장으로'라는 규칙을 두고, 위반(아라비아 숫자 등장, 문장 수 어긋남)을 점검한다.

In [ ]:
# Tip 37: 검열 없음(A) vs <final_review> 강제(B) — 규칙 위반 비교
import re

task = "커피 한 잔이 주는 아침의 활력을 광고 카피로 써줘."
rule = " 규칙: (1) 아라비아 숫자(0-9)를 절대 쓰지 말 것 (2) 정확히 세 문장."

# (A) 규칙만 주고 바로 출력
a = ask(task + rule, reasoning_effort="minimal", max_completion_tokens=300)

# (B) 출력 직전 자체 검열 단계 강제
sys_b = (
    "최종 답을 내기 전에 반드시 <final_review> 태그 안에서 규칙 (1)(2) 준수 여부를"
    " 한 항목씩 점검하라. 위반이 있으면 고쳐서 다시 점검하고, 모두 통과할 때만"
    " </final_review> 뒤에 '최종:' 으로 시작하는 카피만 출력하라."
)
b = ask(task + rule, system=sys_b, reasoning_effort="minimal", max_completion_tokens=600)

def violations(t):
    has_digit = bool(re.search(r"[0-9]", t or ""))
    n_sent = len([s for s in re.split(r"[.!?。]", (t or "")) if s.strip()])
    return {"숫자포함(위반)": has_digit, "문장수": n_sent}

print("(A) 검열 없음     위반체크:", violations(a))
print("(B) final_review  위반체크:", violations(b))
compare("(A) 검열 없음", a, "(B) <final_review> 강제", b)

## 정리 — 이 노트북을 어떻게 읽으면 팁이 '검증'되는가

| 팁 | 무엇을 보면 검증되는가 |
|---|---|
| 3 | 실제 도구 호출 없이도 `<scratchpad>`에 **호출 순서와 입력 출처 계획**이 먼저 나온다 → 나누지 않고도 통제할 수 있다 |
| 4 | (B) 한 번에 처리한 쪽이 세 항목을 **하나의 캠페인 맥락**으로 잘 이어 붙인다; (A)는 조각마다 말투가 제각각이다 |
| 5 | 애매한 입력에서 (A)는 billing/shipping 중 **하나를 억지로 고르고**, (B)는 `Unclassified`로 **안전하게 빠진다** |
| 22 | `keyword_hits` 점수에서 정답지를 겨냥한 (B)가 (A)보다 **키워드 커버리지 점수가 높다** (다시 재현할 수 있는 채점) |
| 24 | AI가 짠 프롬프트에 흔한 표현이 몰려 있고 **도메인 제약이 비어 있다** → 사람이 마무리해야 한다 |
| 27 | (A)는 정보가 없는데도 **처리한 척하거나 넘겨짚고**, (B)는 **부족한 항목을 되묻는다** (첫 오답 전파 차단) |
| 28 | 정답은 둘 다 8인데 압축한 쪽의 **prompt_tokens와 latency가 크게 낮다** → 진짜 성과지표가 줄어든다 |
| 29 | 개요가 **개수를 맞춘 불릿, 정형 구조, 뻔한 헤드라인** → AI 특유의 티, 그대로 납품 금지 |
| 31 | 턴이 늘어도 프롬프트가 **'상태 마크다운 + 명령'으로 일정** → 대화 기록이 폭증하지 않는다 |
| 37 | (A)에는 숫자나 문장 수 **위반이 남고**, (B)는 `<final_review>` 뒤로 위반이 **줄거나 사라진다** |

**한 줄 결론**: 오케스트레이션은 '많이 나누기'가 아니라 '한 에이전트가 잘 계획하고 스스로 검토하게 만들기'이고, 성과는 '느낌'이 아니라 **정답지 채점 + 토큰·응답 시간 실측**으로 판정한다. AI가 만든 결과물은 언제나 **초안**이다.

> gpt-5-nano는 temperature가 고정이라 위의 조절은 모두 **프롬프트 텍스트와 시스템 규칙**으로만 이뤄졌다. 추론 예산이 부족하면 본문이 빌 수 있어, 각 셀은 `reasoning_effort='minimal'`에 넉넉한 `max_completion_tokens`로 맞췄다.